# Lab 12 — Research capstone: observation → evidence → claim

**Dataset:** [ylecun/mnist](https://huggingface.co/datasets/ylecun/mnist).

Research question: **At a small parameter budget, does a nonlinear hidden representation improve held-out accuracy over a linear classifier?**

This notebook is the final Course 1 research artifact: freeze the protocol, reproduce a baseline, intervene once, add a control, create a results table, and write a bounded conclusion.

## 1. Freeze the evaluation contract

**Primary metric:** held-out accuracy.

**Baseline:** linear classifier.

**Intervention:** 32-unit ReLU hidden layer.

**Control:** same MLP trained with shuffled labels.

**Fixed:** dataset subset, preprocessing, optimizer, epochs, seed, and test set.

**Deliverables:** results table, figure, error-analysis notes, reproducibility record, limitations, and next falsifiable experiment.

In [ ]:
!pip -q install datasets torch pandas matplotlib
from datasets import load_dataset
import numpy as np,pandas as pd,torch
from torch import nn
D=load_dataset('ylecun/mnist',split='train[:2500]'); T=load_dataset('ylecun/mnist',split='test[:500]')
def xy(ds):
    X=np.array([np.array(x).reshape(-1) for x in ds['image']],np.float32)/255.; y=np.array(ds['label']); m=y<2
    return torch.tensor(X[m]),torch.tensor(y[m],dtype=torch.float32)
X,y=xy(D); Xt,yt=xy(T); print(X.shape,Xt.shape)

In [ ]:
class Model(nn.Module):
    def __init__(self,h=0):
        super().__init__(); self.net=nn.Linear(784,1) if h==0 else nn.Sequential(nn.Linear(784,h),nn.ReLU(),nn.Linear(h,1))
    def forward(self,x): return self.net(x).squeeze(-1)
def fit(h,labels):
    torch.manual_seed(9); m=Model(h); opt=torch.optim.Adam(m.parameters(),1e-3); loss=nn.BCEWithLogitsLoss()
    for _ in range(6): opt.zero_grad(); loss(m(X),labels).backward(); opt.step()
    with torch.no_grad(): acc=((torch.sigmoid(m(Xt))>.5)==yt.bool()).float().mean().item()
    return acc,sum(p.numel() for p in m.parameters())
rng=np.random.default_rng(9); ysh=torch.tensor(rng.permutation(y.numpy()),dtype=torch.float32)
rows=[]
for name,h,labels in [('linear',0,y),('MLP',32,y),('shuffled_control',32,ysh)]:
    acc,p=fit(h,labels); rows.append([name,h,p,acc])
results=pd.DataFrame(rows,columns=['condition','hidden_units','parameters','test_accuracy']); results

## 2. Produce evidence, not just a number

Save the table to CSV. For a real paper, add confidence intervals or repeated-seed uncertainty. Inspect representative errors and record whether the error looks like class ambiguity, preprocessing failure, or model limitation.

In [ ]:
results.to_csv('results.csv',index=False)
print(results.to_string(index=False))
print('\nReproducibility: seed=9; dataset=ylecun/mnist; train subset=2500; test subset=500; optimizer=Adam; epochs=6.')

## 3. Answer key and claim discipline

**Observation:** report the measured table without interpretation.

**Interpretation:** explain whether the direction is consistent with the hypothesis.

**Claim:** state only what the protocol supports. A small binary MNIST subset cannot establish a universal law about nonlinear networks.

A strong final paragraph has the form: **result → mechanism-consistent interpretation → limitation → next test**.

### Next falsifiable experiment
Match parameter counts more tightly, repeat for at least three seeds, and test whether the advantage survives a distribution shift. Release the code, configuration, dataset provenance, and raw result table with the report.